# Noah Miller 

**Cluster 5 Stacking Model**

**Group 2** | Cluster 5: 432 companies, 3 bankrupt (0.69% rate)

Note: extremely low bankruptcy rate with very few positive samples requires careful handling of class imbalance.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [9]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class.
    TT = correctly predicted bankrupt, TF = bankrupt predicted as non-bankrupt."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    """Check if < 20% of predictions are bankrupt."""
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    """Print evaluation metrics."""
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f'--- {label} ---')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [10]:
# Load cluster 5 data
df5 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_5.csv'))
y5 = df5['Bankrupt?'].values
X5 = df5.drop(columns=['Bankrupt?']).values
feature_names_5 = df5.drop(columns=['Bankrupt?']).columns.tolist()

print(f'Cluster 5: {X5.shape[0]} samples, {X5.shape[1]} features')
print(f'Bankrupt: {y5.sum()} ({y5.mean():.4f})')

Cluster 5: 432 samples, 95 features
Bankrupt: 3 (0.0069)


## Feature Selection

In [11]:
# Feature selection
mi_scores_5 = mutual_info_classif(X5, y5, random_state=RANDOM_STATE)
mi_series_5 = pd.Series(mi_scores_5, index=feature_names_5).sort_values(ascending=False)

# Fewer features — 3 positive samples can't support 20 features
N_FEATURES_5 = 5
top_features_5 = mi_series_5.head(N_FEATURES_5).index.tolist()
X5_sel = df5[top_features_5].values

print(f'Selected {N_FEATURES_5} features for Cluster 5:')
for i, (feat, score) in enumerate(mi_series_5.head(N_FEATURES_5).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

Selected 5 features for Cluster 5:
   1. Contingent liabilities/Net worth                         MI = 0.0419
   2. Net Value Per Share (C)                                  MI = 0.0196
   3. Net Value Per Share (A)                                  MI = 0.0194
   4. Retained Earnings to Total Assets                        MI = 0.0189
   5. Net Value Per Share (B)                                  MI = 0.0182


## Cross-Validation

In [12]:
# Cross-validate with SMOTE + threshold tuning
# Use 3-fold stratified CV (each fold gets ~1 positive)
cv5 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
y5_cv_probas = np.zeros(len(y5))

for fold, (train_idx, val_idx) in enumerate(cv5.split(X5_sel, y5)):
    X_train_cv, X_val_cv = X5_sel[train_idx], X5_sel[val_idx]
    y_train_cv, y_val_cv = y5[train_idx], y5[val_idx]
    
    n_pos = int(y_train_cv.sum())
    if n_pos >= 2:
        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(2, n_pos - 1),
                      sampling_strategy=min(0.3, 1.0))
        X_train_res, y_train_res = smote.fit_resample(X_train_cv, y_train_cv)
    else:
        X_train_res, y_train_res = X_train_cv, y_train_cv
    
    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=2, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('et', ExtraTreesClassifier(n_estimators=100, max_depth=2, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ],
        final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
        cv=min(3, max(2, int(y_train_res.sum()))),
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_res, y_train_res)
    y5_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={y_val_cv.sum()}, avg proba bankrupt={y5_cv_probas[val_idx].mean():.4f}')

# Threshold tuning
print('\n--- Threshold Sweep ---')
best_thresh_5, best_acc_5 = 0.5, 0.0
for thresh in np.arange(0.02, 0.50, 0.01):
    preds = (y5_cv_probas >= thresh).astype(int)
    acc = custom_accuracy(y5, preds)
    _, rate = sparsity_check(preds)
    if rate < 0.20 and acc >= best_acc_5:
        best_acc_5 = acc
        best_thresh_5 = thresh
    if thresh in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        print(f'  thresh={thresh:.2f}: acc={acc:.4f}, sparsity={rate:.4f}')

print(f'\nBest threshold: {best_thresh_5:.2f}')
y5_cv_preds = (y5_cv_probas >= best_thresh_5).astype(int)
evaluate_model(y5, y5_cv_preds, f'Cluster 5 — 3-Fold CV (thresh={best_thresh_5:.2f})')

Fold 1: val positives=1, avg proba bankrupt=0.1377
Fold 2: val positives=1, avg proba bankrupt=0.1437
Fold 3: val positives=1, avg proba bankrupt=0.1237

--- Threshold Sweep ---

Best threshold: 0.15
--- Cluster 5 — 3-Fold CV (thresh=0.15) ---
Custom Accuracy (TT/(TT+TF)): 1.0000
Sparsity: 0.0694 (PASS)
Confusion Matrix:
[[402  27]
 [  0   3]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97       429
           1       0.10      1.00      0.18         3

    accuracy                           0.94       432
   macro avg       0.55      0.97      0.57       432
weighted avg       0.99      0.94      0.96       432



(np.float64(1.0), np.float64(0.06944444444444445))

## Train Final Model & Save

In [13]:
# Train final model for cluster 5 with moderate SMOTE
smote_5 = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(2, int(y5.sum()) - 1),
                sampling_strategy=min(0.3, 1.0))
X5_res, y5_res = smote_5.fit_resample(X5_sel, y5)
print(f'After SMOTE: {len(X5_res)} samples (was {len(X5_sel)})')
print(f'Class distribution: {pd.Series(y5_res).value_counts().to_dict()}')

stacker_5 = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=2, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('et', ExtraTreesClassifier(n_estimators=100, max_depth=2, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_5.fit(X5_res, y5_res)

# Training accuracy with tuned threshold
y5_train_proba = stacker_5.predict_proba(X5_sel)[:, 1]
y5_train_pred = (y5_train_proba >= best_thresh_5).astype(int)
evaluate_model(y5, y5_train_pred, f'Cluster 5 — Training (thresh={best_thresh_5:.2f})')

THRESHOLD_5 = best_thresh_5

After SMOTE: 557 samples (was 432)
Class distribution: {0: 429, 1: 128}
--- Cluster 5 — Training (thresh=0.15) ---
Custom Accuracy (TT/(TT+TF)): 1.0000
Sparsity: 0.0463 (PASS)
Confusion Matrix:
[[412  17]
 [  0   3]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       429
           1       0.15      1.00      0.26         3

    accuracy                           0.96       432
   macro avg       0.57      0.98      0.62       432
weighted avg       0.99      0.96      0.97       432



In [14]:
# Save cluster 5 model with threshold
model_5_info = {
    'model': stacker_5,
    'features': top_features_5,
    'n_features': N_FEATURES_5,
    'threshold': THRESHOLD_5,
    'cluster_id': 5,
    'model_type': 'stacking',
    'member': 'Noah Miller'
}
joblib.dump(model_5_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_5_model.joblib'))
print(f'Cluster 5 model saved. Features: {N_FEATURES_5}, Threshold: {THRESHOLD_5:.2f}')

Cluster 5 model saved. Features: 5, Threshold: 0.15


## Summary Cluster 5

| Metric | Original (N=20) | Final (N=5) |
|--------|-----------------|-------------|
| CV Custom Accuracy | 0% (0/3) | **100% (3/3)** |
| CV Sparsity | 0.46% PASS | **6.9% PASS** |
| Train Custom Accuracy | 100% | **100%** |
| Train Sparsity | — | 4.6% PASS |
| Feature Score | 0.60 | **0.90** |
| Est. Rank Score | — | **0.970** |

**Key changes to reduce overfitting:**
- Reduced features from 20 → 5 (appropriate for 3 positive samples)
- Simplified base models: depth 2, stronger regularization
- Moderate SMOTE (30% ratio instead of 1:1)
- Probability threshold tuning (0.15) instead of default 0.5 cutoff

### Saved Artifacts
- `models/subgroup_models/cluster_5_model.joblib` (includes threshold=0.15)